# Block flow matching language model (`variant: block_flow_matching`)

<img src="../../assets/explainers/block_flow_matching.gif" width="100%">

**The path**: the cosine schedule $\kappa(t) = 1-\cos(\pi t/2)$ against the linear one - characters appear slowly, then all at once. **Training**: the same two-stream block-causal layout as block diffusion, one flow time per block. **Generation**: blocks left to right; inside each block Euler steps $t: 0\to 1$ where a masked character jumps with the exact path probability $(\kappa(t')-\kappa(t))/(1-\kappa(t))$.

## Intuition

Block flow matching is to flow matching what block diffusion is to diffusion: autoregressive across blocks,
a discrete masking **flow** within the current block. The flow view makes two choices explicit: the
path $\kappa(t)$ (how fast tokens become data) and the integration rule.

## Math

$$p_t(x^i \mid x_1^i) = \kappa(t)\,\delta_{x_1^i} + (1-\kappa(t))\,\delta_{[\mathrm{M}]},\qquad
  \text{rate } \frac{\kappa'(t)}{1-\kappa(t)}\ \begin{cases}\frac{1}{1-t} & \text{linear}\\[2pt]
  \frac{\pi}{2}\tan\frac{\pi t}{2} & \text{cosine}\end{cases}$$

$$\mathcal{L} = \sum_b \mathbb{E}_{t_b}\Big[\frac{\kappa'(t_b)}{1-\kappa(t_b)}\sum_{i\in b,\ \text{masked}}
  -\log p_\theta(x_1^i \mid x_{t_b}^{(b)}, x^{(<b)}, t_b)\Big].$$

The clean stream sits at the end of the flow ($t = 1$). Each Euler step uses the exact transition
$P(\text{masked at } t \text{ is data at } t') = \frac{\kappa(t')-\kappa(t)}{1-\kappa(t)}$, so the product
over steps reaches 1 at $t=1$ for any step count.

## In the code

| File | What |
|---|---|
| `models_block_flow_matching/path.py` | `MaskingPath` (`kappa`, `rate`, `unmask_probability`), `corrupt_blocks` |
| `.../model.py` | `BlockFlowMatchingLM` (two streams, clean at t=1), collate |
| `.../loss.py`, `.../sample.py`, `.../metrics.py` | rate-weighted CE, block Euler sampler, `masked_accuracy` |
| config | `model.fm_schedule: linear | cosine`, `sample.sampling_steps` = Euler steps per block |

In [ ]:
import sys
from pathlib import Path

# Run from anywhere inside the repo: put src/ on the path if hale_llm is not installed.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import HTML, display

import hale_llm.models  # noqa: F401  - registers every paradigm
from hale_llm.core.registry import get_loss, get_sampler, get_variant
from hale_llm.dataloader.collate import get_collate
from hale_llm.visualization.explainers import CORPUS, get_explainer

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
print("training corpus:")
for sentence in CORPUS:
    print("  ", repr(sentence))

## Train a toy model

Blocks of 8, cosine path (`fm_schedule: cosine`). The explainer's recipe trains a ~100k-parameter model on the four corpus sentences
in a few seconds on a CPU - small enough to inspect every number, trained enough to show real
behaviour.

In [ ]:
explainer = get_explainer("block_flow_matching")
run = explainer.train()
tok, model, cfg = run.tokenizer, run.model, run.cfg
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("config:", {k: v for k, v in cfg.model.model_dump().items()
                  if k in ("arch", "d_model", "n_layers", "attn_type", "use_time_cond",
                           "block_size", "n_mtp_heads", "fm_schedule")})

fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.semilogy(run.losses, color="#264653")
ax.set(xlabel="step", ylabel="training loss", title="toy training")
plt.show()

## Linear vs cosine paths

In [ ]:
import numpy as np
from hale_llm.models.models_block_flow_matching import MaskingPath, corrupt_blocks

ts = torch.linspace(0, 0.98, 99)
fig, ax = plt.subplots(1, 2, figsize=(9, 2.6))
for name, color in (("linear", "#264653"), ("cosine", "#E76F51")):
    path = MaskingPath(name)
    ax[0].plot(ts, path.kappa(ts), color=color, label=name)
    ax[1].plot(ts, path.rate(ts), color=color, label=name)
ax[0].set(title="κ(t): share of data tokens", xlabel="t"); ax[0].legend()
ax[1].set(title="rate κ′/(1−κ) = loss weight", xlabel="t", yscale="log"); ax[1].legend()
plt.show()

path = model.path
steps = torch.linspace(0, 1, 9).tolist()
survive = np.prod([1 - path.unmask_probability(a, b) for a, b in zip(steps[:-1], steps[1:])])
print("P(still masked after 8 exact Euler steps) =", survive)

## A training example: per-block flow times

In [ ]:
torch.manual_seed(2)
x_t, masked, t = corrupt_blocks(run.ids, 8, tok.mask_token_id, model.path)
for b in range(6):
    seg = "".join("_" if masked[0, i] else run.text[i] for i in range(b * 8, b * 8 + 8))
    tb = float(t[0, b * 8])
    print(f"block {b}: t={tb:.2f} κ={float(model.path.kappa(torch.tensor(tb))):.2f}  {seg!r}")
with torch.no_grad():
    print("loss:", float(get_loss("block_flow_matching")(model, get_collate("block_flow_matching")(run.ids, tok))))

In [ ]:
def validity(steps_list, seeds=24, temperature=1.0):
    """Share of samples that are exactly one of the training sentences, per sampling-step budget."""
    prompt = run.ids[:, :8]
    sampler = get_sampler(cfg.variant)
    out = {}
    for steps in steps_list:
        ok = 0
        for seed in range(seeds):
            torch.manual_seed(seed)
            ids = sampler(model, tok, prompt, max_new_tokens=40, temperature=temperature, sampling_steps=steps)
            ok += tok.decode(ids[0]) in CORPUS
        out[steps] = ok / seeds
    return out

In [ ]:
print("valid share vs Euler steps per block:", validity([1, 2, 4, 8]))

In [ ]:
trace = explainer.trace(run, seed=0)
assert torch.equal(trace.ids, explainer.reference(run, seed=0))
for s in trace.steps:
    canvas = "".join("_" if v == tok.mask_token_id else tok.token(v) for v in s.x)
    tag = "fill" if s.final_fill else f"t {s.t_from:.3f}→{s.t_to:.3f}"
    print(f"block {s.block} {tag:15} p {s.p_reveal:.2f} +{len(s.revealed)}  {canvas}")

In [ ]:
# Re-render the animation from this notebook's model. The committed copy shown at the top is
# assets/explainers/block_flow_matching.gif (regenerate all of them with `hale-llm-explain`).
from PIL import Image

out = ROOT / "notebooks" / "models" / "outputs" / "block_flow_matching.gif"  # git-ignored
explainer.build(out, run=run)
with Image.open(out) as gif:
    print(out.relative_to(ROOT), f"{gif.n_frames} frames, {gif.size[0]}x{gif.size[1]},",
          f"{out.stat().st_size / 1024:.0f} KB")
# In Jupyter: display(HTML('<img src="outputs/block_flow_matching.gif" width="100%">'))

## Takeaways

* Same two-stream, block-causal machinery as block diffusion; the within-block generator is a flow with a
  chosen path and an exact Euler transition.
* Cosine reveals few tokens early (when the context is thin) and most near $t=1$.

**Exercise.** Train a `linear` model (`model={"fm_schedule": "linear", ...}`) and compare `validity` at
1 and 2 Euler steps per block.